In [7]:
import pickle
import json
import numpy as np

In [11]:
def object_data_to_3d_vertices_scores(object_data, image_width=2160, image_height=1080):    
    all_frame_vertices = []
    all_frame_scores = []
    
    for frame_id in range(len(object_data)):
        frame_data = object_data[frame_id].copy()
                
        object_scores = frame_data['scores']
        all_frame_scores.append(object_scores.tolist())
        
        object_verts2d = frame_data['pred_verts2d']
        object_verts3d = frame_data['pred_verts3d']
    
        object_verts2d = object_verts2d.copy()
        num_objects = object_verts2d.shape[0]
        object_verts2d = object_verts2d.reshape(num_objects, 3, 8)
        object_verts2d = object_verts2d.transpose(0, 2, 1)
        
        # pixel coords -> image coords
        object_verts2d[:, :, 0] /= image_width
        object_verts2d[:, :, 1] /= image_height
        object_verts2d = object_verts2d[:, :, :2]  
        
        # image coords -> spherical coords
        object_verts2d[:, :, 0] = (object_verts2d[:, :, 0] - 0.5) * 2 * np.pi + np.pi/2
        object_verts2d[:, :, 1] = (object_verts2d[:, :, 1] - 0.5) * np.pi

        # spherical coords -> world coords
        r = object_verts3d[:, :, 2] * 0.4 # use z value as radius
        z = r * np.sin(object_verts2d[:, :, 1])
        x = r * np.cos(object_verts2d[:, :, 1]) * np.sin(object_verts2d[:, :, 0]) * -1
        y = r * np.cos(object_verts2d[:, :, 1]) * np.cos(object_verts2d[:, :, 0])
        all_obj_verts = np.stack([x, y, z], axis=-1)
        all_frame_vertices.append(all_obj_verts.tolist())

    return all_frame_vertices, all_frame_scores

In [12]:
# read pickled data
joint_pkl_path = '/home/max/Documents/RA/egowholemocap/work_dirs/mo2cap2_single/outputs_refined.pkl'
object_pkl_path = '/home/max/Documents/RA/ovmono3d/output/insta360_unprocessed/detections_reordered.pkl'

with open(joint_pkl_path, 'rb') as f:
        joint_data = pickle.load(f)
        
if object_pkl_path:
        with open(object_pkl_path, 'rb') as f:
                object_data = pickle.load(f)
                object_vertices, object_scores = object_data_to_3d_vertices_scores(object_data)

In [14]:
len(joint_data), len(object_vertices), len(object_scores)

(784, 497, 497)

In [21]:
# each frame contains:
#   - frame_id
#   - joint_poses: list of [x, y, z] positions
#   - objects:
#     - object_id
#     - score
#     - vertices: list of [x, y, z] vertices

if object_vertices:
    frame_count = min(len(joint_data), len(object_vertices), len(object_scores))
else:
    frame_count = len(joint_data)
    
combined_list = []
for frame_idx in range(frame_count):
    frame_pose = joint_data[frame_idx]['mo2cap2_pred_motion']
    frame_object_verts = object_vertices[frame_idx]
    frame_object_scores = object_scores[frame_idx]
    
    # Create objects list with individual object dictionaries
    objects_list = []
    for obj_idx in range(len(frame_object_scores)):
        objects_list.append({
            "object_id": obj_idx,
            "score": frame_object_scores[obj_idx],
            "vertices": frame_object_verts[obj_idx]
        })
    
    frame_dict = {
        "frame_id": frame_idx,
        "joint_poses": frame_pose.tolist(),
        "objects": objects_list
    }
    combined_list.append(frame_dict)

In [20]:
with open("/home/max/UnityProjects/ar_tut/Assets/Animations/mo2cap2.json", "w", encoding="utf-8") as f:
    json.dump(combined_list, f)